In [22]:
from pathlib import Path
import numpy as np
import pandas as pd
from pedalboard.io import AudioFile
from pedalboard import Pedalboard, Reverb, Chorus

In [23]:
base_path = Path("..")
raw_path = base_path / "data" / "raw"
processed_path = base_path / "data" / "processed"
metadata_path = base_path / "data" / "metadata.csv"
data_path = raw_path / "egfxset" / "Clean"

raw_path, processed_path, metadata_path, data_path

(WindowsPath('../data/raw'),
 WindowsPath('../data/processed'),
 WindowsPath('../data/metadata.csv'),
 WindowsPath('../data/raw/egfxset/Clean'))

#### Диапазоны параметров

In [24]:
PARAM_RANGES = {
    "room_size": (0.0, 1.0),
    "wet_level": (0.0, 0.6),
    "rate_hz": (0.5, 5.0),
    "depth": (0.0, 1.0),
}

PARAM_RANGES

{'room_size': (0.0, 1.0),
 'wet_level': (0.0, 0.6),
 'rate_hz': (0.5, 5.0),
 'depth': (0.0, 1.0)}

#### Случайный выбор параметров

In [25]:
def sample_parameters():
    return {
        "room_size": np.random.uniform(0.0, 1.0),
        "wet_level": np.random.uniform(0.0, 0.6),
        "rate_hz": np.random.uniform(0.5, 5.0),
        "depth": np.random.uniform(0.0, 1.0),
    }

#### Нормализация параметров

In [26]:
def normalize_params(params):
    return {
        "room_size": params["room_size"],
        "wet_level": params["wet_level"] / 0.6,
        "rate_hz": (params["rate_hz"] - 0.5) / 4.5,
        "depth": params["depth"],
    }

def denormalize_params(params):
    return {
            "room_size": params["room_size"],
            "wet_level": params["wet_level"] * 0.6,
            "rate_hz": params["rate_hz"] * 4.5 + 0.5,
            "depth": params["depth"],
        }

#### Обработка аудиосигнала

In [27]:
def apply_fx(audio, sample_rate, params):

    board = Pedalboard([
        Reverb(
            room_size=params["room_size"],
            wet_level=params["wet_level"]
        ),

        Chorus(
            rate_hz=params["rate_hz"],
            depth=params["depth"]
        )
    ])

    wet_audio = board(audio, sample_rate)

    return wet_audio

#### Чтение файлов

In [29]:
audio_files = list(data_path.rglob("*.wav"))
print(len(audio_files))
print(audio_files[:5])

690
[WindowsPath('../data/raw/egfxset/Clean/Bridge/1-0.wav'), WindowsPath('../data/raw/egfxset/Clean/Bridge/1-1.wav'), WindowsPath('../data/raw/egfxset/Clean/Bridge/1-10.wav'), WindowsPath('../data/raw/egfxset/Clean/Bridge/1-11.wav'), WindowsPath('../data/raw/egfxset/Clean/Bridge/1-12.wav')]


In [33]:
def get_source_id(audio_path):
    pickup = audio_path.parent.name
    file_id = audio_path.stem

    return f"{pickup}_{file_id}"

variants = 5
test_path = audio_files[0]
source_id = get_source_id(test_path)

for i in range(variants):
    params = sample_parameters()
    norm_params = normalize_params(params)

    print(source_id, i, params, norm_params)

Bridge_1-0 0 {'room_size': 0.3334135442705818, 'wet_level': 0.1335699799690008, 'rate_hz': 0.7534451162287503, 'depth': 0.8846004870585061} {'room_size': 0.3334135442705818, 'wet_level': 0.222616633281668, 'rate_hz': 0.056321136939722294, 'depth': 0.8846004870585061}
Bridge_1-0 1 {'room_size': 0.06636683384900299, 'wet_level': 0.19741766950849152, 'rate_hz': 1.9497748543305717, 'depth': 0.08924762372182649} {'room_size': 0.06636683384900299, 'wet_level': 0.3290294491808192, 'rate_hz': 0.3221721898512382, 'depth': 0.08924762372182649}
Bridge_1-0 2 {'room_size': 0.5033822931073306, 'wet_level': 0.3303694120762371, 'rate_hz': 1.033106769755932, 'depth': 0.23428226399938423} {'room_size': 0.5033822931073306, 'wet_level': 0.5506156867937285, 'rate_hz': 0.11846817105687377, 'depth': 0.23428226399938423}
Bridge_1-0 3 {'room_size': 0.0612067697782267, 'wet_level': 0.3049199871754814, 'rate_hz': 3.4912117280454655, 'depth': 0.14273435939684542} {'room_size': 0.0612067697782267, 'wet_level': 0.5

In [31]:
test_output_path = processed_path / "test_wet"
test_output_path.mkdir(parents=True, exist_ok=True)

In [36]:
with AudioFile(str(test_path)) as f:
    audio = f.read(f.frames)
    sample_rate = f.samplerate

print(audio.shape)
print(sample_rate)

(1, 240000)
48000


In [37]:
params = sample_parameters()

wet_audio = apply_fx(
    audio=audio,
    sample_rate=sample_rate,
    params=params
)

output_file = processed_path / "test_wet.wav"

with AudioFile(
    str(output_file),
    "w",
    sample_rate,
    wet_audio.shape[0]
) as f:
    f.write(wet_audio)

print("Saved:", output_file)
print("Wet shape:", wet_audio.shape)
print("Params:", params)

Saved: ..\data\processed\test_wet.wav
Wet shape: (1, 240000)
Params: {'room_size': 0.5788953462488206, 'wet_level': 0.526384726550396, 'rate_hz': 3.4534842468105182, 'depth': 0.8699849199814937}


#### Проверка

In [41]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [40]:
N_VARIANTS = 5

rows = []

test_path = audio_files[0]
source_id = get_source_id(test_path)

with AudioFile(str(test_path)) as f:
    audio = f.read(f.frames)
    sample_rate = f.samplerate

test_output_path = processed_path / "test_wet"
test_output_path.mkdir(parents=True, exist_ok=True)

for i in range(N_VARIANTS):
    params = sample_parameters()
    norm_params = normalize_params(params)

    wet_audio = apply_fx(
        audio=audio,
        sample_rate=sample_rate,
        params=params
    )

    output_file = test_output_path / f"{source_id}_v{i}.wav"

    with AudioFile(
        str(output_file),
        "w",
        sample_rate,
        wet_audio.shape[0]
    ) as f:
        f.write(wet_audio)

    row = {
        "source_id": source_id,
        "wet_path": str(output_file),

        "room_size": params["room_size"],
        "wet_level": params["wet_level"],
        "rate_hz": params["rate_hz"],
        "depth": params["depth"],

        "room_size_norm": norm_params["room_size"],
        "wet_level_norm": norm_params["wet_level"],
        "rate_hz_norm": norm_params["rate_hz"],
        "depth_norm": norm_params["depth"],
    }

    rows.append(row)

df = pd.DataFrame(rows)

df.shape


(5, 10)

#### Сборка датасета

In [42]:
def generate_dataset(audio_files, output_path, n_variants=5):
    rows = []

    output_path.mkdir(parents=True, exist_ok=True)

    for audio_path in audio_files:
        source_id = get_source_id(audio_path)

        with AudioFile(str(audio_path)) as f:
            audio = f.read(f.frames)
            sample_rate = f.samplerate

        for i in range(n_variants):
            params = sample_parameters()
            norm_params = normalize_params(params)

            wet_audio = apply_fx(
                audio=audio,
                sample_rate=sample_rate,
                params=params
            )

            output_file = output_path / f"{source_id}_v{i}.wav"

            with AudioFile(
                str(output_file),
                "w",
                sample_rate,
                wet_audio.shape[0]
            ) as f:
                f.write(wet_audio)

            row = {
                "source_id": source_id,
                "wet_path": str(output_file),

                "room_size": params["room_size"],
                "wet_level": params["wet_level"],
                "rate_hz": params["rate_hz"],
                "depth": params["depth"],

                "room_size_norm": norm_params["room_size"],
                "wet_level_norm": norm_params["wet_level"],
                "rate_hz_norm": norm_params["rate_hz"],
                "depth_norm": norm_params["depth"],
            }

            rows.append(row)

    return pd.DataFrame(rows)

In [44]:
#ласт тест

np.random.seed(42)

wet_path = processed_path / "wet"

test_df = generate_dataset(
    audio_files=audio_files[:3],
    output_path=wet_path,
    n_variants=5
)

test_df.shape

(15, 10)